In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
import json
import pandas as pd
import os.path as osp
from glob import glob
from data_utils import parse_round, gameid_order

sys.path.append('../')
from configuration_template import Config

config = Config()

model_version: Qwen2
model_type: Qwen
/home/alesau/mllm/mllm/PATH


In [2]:
# Data comes from this paper: https://www.aclweb.org/anthology/P19-1059/
files = glob(osp.join(config.colorgrid_data, 'examples', 'games', 'json', 'colorGrids', 'raw', '*/*'))

In [3]:
# collect game data

all_games = dict()

for file in files:
    # every file is a game

    with open(file, "r") as f:
        game = json.load(f)

    records = game["records"]
    gameid = game["gameid"]
    for i in range(len(records)):
        records[i]["gameid"] = gameid

    all_games[gameid] = records

In [4]:
# collect, parse and filter rounds for all games

# store all rounds in flat list
all_rounds = []
for rounds in all_games.values():
    all_rounds += rounds

# reformat rounds
parsed_rounds = [parse_round(r) for r in all_rounds]

# filter out invalid rounds
valid_rounds = [
    p
    for p in parsed_rounds
    if (p["success"] != None)  # keep only if states included
    if (len(p["utterances"]) > 0)  # keep only if utterances included
]

print('rounds (total):', len(parsed_rounds))
print('valid: (valid):', len(valid_rounds))

rounds (total): 10961
valid: (valid): 10925


In [5]:
# build data frame (with rounds in separate rows)
rounds_df = pd.DataFrame(valid_rounds)

# to reproduce our order:
rounds_df = rounds_df.sort_values(by=['gameid', 'roundNum'], key=lambda column: column.map(lambda e: gameid_order.get(e))).reset_index(drop=True)

rounds_df['round_id'] = rounds_df.apply(lambda x: f'{x.gameid}_{x.roundNum}', axis=1)
rounds_df['n_utterances'] = rounds_df.utterances.map(len)  # number of utterances in this round
game_idx = pd.factorize(rounds_df['gameid'])[0] + 1  # offset to match roundNum (starts with 1)
rounds_df['game_idx'] = game_idx

# reorder columns
rounds_df = rounds_df[[
    'gameid', 
    'game_idx',
    'roundNum', 
    'round_id',
    'condition', 
    'success',
    'utterances', 
    'n_utterances', 
    'objs', 
    'target',
    'speaker_order', 
    'listener_order', 
    'listener_clicked', 
]]

In [7]:
rounds_df

,gameid,game_idx,roundNum,round_id,condition,success,utterances,n_utterances,objs,target,speaker_order,listener_order,listener_clicked
0,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,1,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_1,FAR,True,"[(speaker, green top left corner, 1528144087280)]",1,"[{'shapes': [{'color': [322, 21, 50]}, {'color...",1,"[2, 1, 0]","[1, 0, 2]",0
1,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,2,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_2,SPLIT,True,"[(speaker, pink in the bottom left, 1528144112...",1,"[{'shapes': [{'color': [55, 5, 50]}, {'color':...",2,"[2, 1, 0]","[0, 2, 1]",1
2,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,3,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_3,SPLIT,False,"[(speaker, dark orange top right corner, 15281...",1,"[{'shapes': [{'color': [206, 56, 50]}, {'color...",0,"[0, 1, 2]","[1, 2, 0]",1
3,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,4,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_4,FAR,True,"[(speaker, green in the middle square top row,...",1,"[{'shapes': [{'color': [294, 7, 50]}, {'color'...",0,"[1, 2, 0]","[2, 1, 0]",2
4,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,5,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_5,SPLIT,True,"[(speaker, top right corner red, 1528144188447)]",1,"[{'shapes': [{'color': [227, 76, 50]}, {'color...",1,"[0, 2, 1]","[1, 2, 0]",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10920,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,56,6068-c8135454-2875-41d9-918c-60c34fc9460c_56,CLOSE,True,"[(speaker, top right not darkest not lightest ...",1,"[{'shapes': [{'color': [121, 13, 50]}, {'color...",1,"[0, 1, 2]","[2, 0, 1]",2
10921,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,57,6068-c8135454-2875-41d9-918c-60c34fc9460c_57,FAR,True,"[(speaker, Top middle green, 1526435245005)]",1,"[{'shapes': [{'color': [195, 34, 50]}, {'color...",1,"[2, 1, 0]","[1, 2, 0]",0
10922,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,58,6068-c8135454-2875-41d9-918c-60c34fc9460c_58,FAR,True,"[(speaker, middle right pink, 1526435266994)]",1,"[{'shapes': [{'color': [187, 64, 50]}, {'color...",2,"[0, 1, 2]","[2, 0, 1]",0
10923,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,59,6068-c8135454-2875-41d9-918c-60c34fc9460c_59,CLOSE,True,"[(speaker, Bottom middle is the boldest yellow...",1,"[{'shapes': [{'color': [48, 19, 50]}, {'color'...",0,"[1, 0, 2]","[1, 2, 0]",2


In [11]:
def create_condition_subset(data, condition):
    if isinstance(condition, str):
        condition = [condition]

    return data[data["condition"].isin(condition)]

print(f"Total number of rounds: {len(rounds_df)}")
far_subset = create_condition_subset(rounds_df, "FAR")
print(f"Number of rounds in 'FAR' condition: {len(far_subset)}")
close_subset = create_condition_subset(rounds_df, "CLOSE")
print(f"Number of rounds in 'CLOSE' condition: {len(close_subset)}")
split_subset = create_condition_subset(rounds_df, ["SPLIT"])
print(f"Number of rounds in 'SPLIT' conditions: {len(split_subset)}")
far_split_subset = create_condition_subset(rounds_df, ["FAR", "SPLIT"])
print(f"Number of rounds in 'FAR' and 'SPLIT' conditions: {len(far_split_subset)}")

Total number of rounds: 10925
Number of rounds in 'FAR' condition: 3635
Number of rounds in 'CLOSE' condition: 3654
Number of rounds in 'SPLIT' conditions: 3636
Number of rounds in 'FAR' and 'SPLIT' conditions: 7271


In [8]:
out_path = osp.join(config.data_dir, 'color_grid_data.json')
rounds_df.to_json(out_path, orient='records')

In [12]:
out_path = osp.join(config.data_dir, 'color_grid_data_far.json')
far_subset.to_json(out_path, orient='records')